# TSE Analysis 2026 — Ingestão de Logs da Urna (`fact_urna_logs`)

Este notebook implementa de forma didática e direta a ingestão da tabela **`fact_urna_logs`** a partir dos arquivos `.jez` (`logd.dat`) das **Eleições 2026**:
1. **Extração parcial via HTTP Range** de um arquivo `.jez` real do 1º turno de 2026 no Acre (`o03220ac0101500020088-log.jez`) diretamente do pacote oficial do TSE (`arqurnatot/bu_imgbu_logjez_rdv_vscmr_2026_1t_AC.zip`), baixando apenas ~68 KB em vez de 393 MB.
2. **Schema 1:1 (`fact_urna_logs`)**: Preserva as colunas originais separadas por tabulação (`\t`) do `logd.dat` (`raw_timestamp`, `log_level`, `urna_serial`, `application`, `message_text` com números de 5+ dígitos mascarados como `<ID_REDACTED>`), omitindo `mac_code` para evitar ~35 GB de entropia hexadecimal aleatória e permitir alta taxa de compressão por dicionário em Parquet ZSTD.
3. **Inspeção de `HEAD` e `TAIL`** e validação de integridade física em **Polars**, **PyArrow (Parquet ZSTD)** e **DuckDB**.

In [1]:
import importlib
import importlib.util
import inspect
import os
import shutil
import stat
import subprocess
import sys
from pathlib import Path

_REQUIRED_PACKAGES = (
    ("kagglehub", "kagglehub>=0.4.0"),
    ("polars", "polars"),
    ("pyarrow", "pyarrow"),
    ("duckdb", "duckdb"),
    ("httpx", "httpx"),
)
if globals().get("__name__") == "__main__" and not os.getenv("PYTEST_CURRENT_TEST"):
    for _mod_name, _pip_name in _REQUIRED_PACKAGES:
        if importlib.util.find_spec(_mod_name) is None:
            subprocess.check_call(
                [sys.executable, "-m", "pip", "install", "-q", "-U", _pip_name]
            )
            importlib.invalidate_caches()
    import kagglehub as _kh_check

    if "output_dir" not in inspect.signature(_kh_check.dataset_download).parameters:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", "-U", "kagglehub>=0.4.0"]
        )
        importlib.invalidate_caches()
        sys.modules.pop("kagglehub", None)
        import kagglehub as _kh_check

# Contrato único de estado e ciclo de vida via KaggleHub SDK (Kaggle e Google Colab)
DATASET_HANDLE = os.getenv("KAGGLE_DATASET_HANDLE", "miolodados/tse-eleicoes-dados")
DATASET_PATH = os.getenv("DATASET_PATH", "/data/export/tse-eleicoes-dados")
STAGING_PATH = os.getenv("STAGING_PATH", "/tmp/tse-eleicoes-staging")
ALLOW_COLD_START = True


def is_directory_empty(path: Path) -> bool:
    """Verifica se DATASET_PATH está efetivamente vazio (ignorando arquivos ocultos de sistema)."""
    if not path.exists():
        return True
    return not any(item for item in path.iterdir() if not item.name.startswith("."))


def ensure_writable_tree(root: Path) -> None:
    """Garante permissão de escrita em árvores copiadas de mounts somente-leitura (/kaggle/input)."""
    if not root.exists():
        return
    root.chmod(stat.S_IRWXU | stat.S_IRGRP | stat.S_IXGRP | stat.S_IROTH | stat.S_IXOTH)
    for dirpath, dirnames, filenames in os.walk(root):
        base = Path(dirpath)
        for dname in dirnames:
            dp = base / dname
            if not dp.is_symlink():
                dp.chmod(stat.S_IRWXU | stat.S_IRGRP | stat.S_IXGRP | stat.S_IROTH | stat.S_IXOTH)
        for fname in filenames:
            fp = base / fname
            if not fp.is_symlink():
                fp.chmod(stat.S_IRUSR | stat.S_IWUSR | stat.S_IRGRP | stat.S_IROTH)


def is_dataset_not_found_error(exc: BaseException) -> bool:
    """Detecta erros de dataset inexistente (404 / Not Found) para permitir reconstrução do zero."""
    msg = f"{type(exc).__name__}: {exc}".lower()
    not_found_markers = (
        "404",
        "not found",
        "does not exist",
        "no dataset found",
        "resource not found",
        "403",
    )
    return any(marker in msg for marker in not_found_markers)


def ensure_kaggle_auth() -> None:
    """Autentica via KaggleHub SDK sem travar execuções headless (batch kernels / CI)."""
    import kagglehub

    has_env_creds = bool(
        os.getenv("KAGGLE_API_TOKEN")
        or (os.getenv("KAGGLE_USERNAME") and os.getenv("KAGGLE_KEY"))
        or (Path.home() / ".kaggle" / "kaggle.json").exists()
        or Path("/root/.config/kaggle/kaggle.json").exists()
    )
    if has_env_creds:
        print("[AUTH] Credenciais do Kaggle detectadas no ambiente.")
        return
    if sys.stdin is not None and sys.stdin.isatty():
        print("Iniciando autenticação manual no Kaggle SDK...")
        kagglehub.login()
    else:
        print(
            "[AUTH] Execução não-interativa detectada; pulando prompt interativo kagglehub.login()."
        )


def bootstrap_dataset(
    handle: str, target_dir: Path, allow_cold_start: bool = False
) -> Path:
    """Se DATASET_PATH estiver vazio, baixa a última versão do dataset do Kaggle ou reconstrói do zero se não existir."""
    target_dir.mkdir(parents=True, exist_ok=True)
    if not is_directory_empty(target_dir):
        print(f"[OK] DATASET_PATH ({target_dir}) já contém dados locais. Pulando download.")
        return target_dir

    print(
        f"[BOOTSTRAP] DATASET_PATH ({target_dir}) está vazio. Baixando '{handle}' via Kaggle SDK..."
    )
    try:
        import kagglehub

        try:
            downloaded_path = Path(
                kagglehub.dataset_download(
                    handle,
                    output_dir=str(target_dir),
                    force_download=True,
                )
            )
        except TypeError as type_err:
            if "output_dir" not in str(type_err):
                raise
            downloaded_path = Path(
                kagglehub.dataset_download(
                    handle,
                    force_download=True,
                )
            )

        if downloaded_path.resolve() != target_dir.resolve() and downloaded_path.exists():
            print(
                f"[SYNC] Copiando arquivos de {downloaded_path} (read-only mount) para {target_dir}..."
            )
            shutil.copytree(downloaded_path, target_dir, dirs_exist_ok=True)
            ensure_writable_tree(target_dir)

        print(f"[OK] Dataset populado com sucesso em {target_dir}.")
    except Exception as exc:
        if allow_cold_start or is_dataset_not_found_error(exc):
            print(
                f"[COLD-START] Dataset '{handle}' não encontrado ou ainda não publicado ({exc}). "
                f"Iniciando reconstrução do zero em {target_dir}."
            )
        else:
            raise RuntimeError(
                f"Falha ao popular DATASET_PATH ({target_dir}) a partir do Kaggle ('{handle}'). "
                "Verifique sua autenticação e conexão de rede."
            ) from exc

    return target_dir


if globals().get("__name__") == "__main__" and not os.getenv("PYTEST_CURRENT_TEST"):
    Path(DATASET_PATH).mkdir(parents=True, exist_ok=True)
    Path(STAGING_PATH).mkdir(parents=True, exist_ok=True)
    ensure_kaggle_auth()
    bootstrap_dataset(
        DATASET_HANDLE, Path(DATASET_PATH), allow_cold_start=ALLOW_COLD_START
    )

import io
import json
import re
import struct
import zipfile
import zlib
from datetime import datetime

import duckdb
import httpx
import polars as pl
import pyarrow.parquet as pq

pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_cols(16)
pl.Config.set_fmt_str_lengths(90)

polars.config.Config

In [2]:
# 1. Funções didáticas de leitura via HTTP Range e parsing 1:1 para fact_urna_logs
TSE_2026_AC_1T_URL = (
    "https://cdn.tse.jus.br/estatistica/sead/eleicoes/eleicoes2026/arqurnatot/"
    "bu_imgbu_logjez_rdv_vscmr_2026_1t_AC.zip"
)

JEZ_NAME_RE = re.compile(
    r"^o(?P<pleito>\d{5})(?P<uf>[a-z]{2})(?P<mun>\d{5})(?P<zona>\d{4})(?P<secao>\d{4})-log\.jez$",
    re.IGNORECASE,
)

FACT_URNA_LOGS_SCHEMA = pl.Schema({
    "election_year": pl.Int16,
    "turn": pl.Int8,
    "uf": pl.String,
    "cd_pleito": pl.Int32,
    "cd_municipio": pl.Int32,
    "nr_zona": pl.Int16,
    "nr_secao": pl.Int32,
    "source_jez_file": pl.String,
    "source_log_member": pl.String,
    "file_seq": pl.Int64,
    "raw_timestamp": pl.Datetime("us"),
    "log_level": pl.String,
    "urna_serial": pl.String,
    "application": pl.String,
    "message_text": pl.String,
})


def fetch_single_jez_from_remote_zip(zip_url: str, jez_index: int = 0) -> tuple[str, bytes]:
    """Lê o Central Directory no final do ZIP remoto via HTTP Range e baixa apenas 1 arquivo .jez."""
    with httpx.Client(timeout=30.0, follow_redirects=True) as client:
        total_size = int(client.head(zip_url).headers["content-length"])
        tail_len = min(total_size, 2 * 1024 * 1024)
        tail = client.get(
            zip_url, headers={"Range": f"bytes={total_size - tail_len}-{total_size - 1}"}
        ).content
        eocd_pos = tail.rfind(b"PK\x05\x06")
        cd_size, cd_offset = struct.unpack("<II", tail[eocd_pos + 12 : eocd_pos + 20])
        cd = client.get(
            zip_url, headers={"Range": f"bytes={cd_offset}-{cd_offset + cd_size - 1}"}
        ).content

        pos = 0
        found = []
        while pos + 46 <= len(cd):
            if cd[pos : pos + 4] != b"PK\x01\x02":
                break
            comp_method = struct.unpack("<H", cd[pos + 10 : pos + 12])[0]
            comp_size, _, fn_len, extra_len, comment_len = struct.unpack(
                "<IIHHH", cd[pos + 20 : pos + 34]
            )
            lhoff = struct.unpack("<I", cd[pos + 42 : pos + 46])[0]
            fname = cd[pos + 46 : pos + 46 + fn_len].decode("latin1")
            pos += 46 + fn_len + extra_len + comment_len
            if fname.lower().endswith(".jez"):
                found.append((fname, comp_method, comp_size, lhoff))
                if len(found) > jez_index:
                    break

        fname, comp_method, comp_size, lhoff = found[jez_index]
        raw_entry = client.get(
            zip_url, headers={"Range": f"bytes={lhoff}-{lhoff + 30 + len(fname) + 256 + comp_size}"}
        ).content
        fn_l, ex_l = struct.unpack("<HH", raw_entry[26:30])
        comp_data = raw_entry[30 + fn_l + ex_l : 30 + fn_l + ex_l + comp_size]
        jez_bytes = zlib.decompress(comp_data, -zlib.MAX_WBITS) if comp_method == 8 else comp_data
        return fname, jez_bytes


def sanitize_message_text(msg: str) -> str:
    """Mascara sequências numéricas de 5+ dígitos (ex.: título de mesário) preservando o texto 1:1."""
    return re.sub(r"\b\d{5,}\b", "<ID_REDACTED>", msg.strip())


def parse_jez_to_fact_urna_logs(
    jez_filename: str, jez_bytes: bytes, election_year: int = 2026, turn: int = 1
) -> tuple[list[str], pl.DataFrame]:
    """Extrai logd.dat em memória e converte 1:1 para o schema de fact_urna_logs."""
    m = JEZ_NAME_RE.match(Path(jez_filename).name)
    if not m:
        raise ValueError(f"Nome de arquivo .jez fora do padrão esperado: {jez_filename}")

    cd_pleito = int(m.group("pleito"))
    uf = m.group("uf").upper()
    cd_municipio = int(m.group("mun"))
    nr_zona = int(m.group("zona"))
    nr_secao = int(m.group("secao"))

    raw_lines_sanitized: list[str] = []
    rows: list[dict[str, object]] = []
    seq = 1

    with zipfile.ZipFile(io.BytesIO(jez_bytes), mode="r") as zf:
        members = sorted(
            [info.filename for info in zf.infolist() if not info.is_dir()],
            key=lambda name: (-(int(re.search(r"\.(\d+)$", name).group(1)) if re.search(r"\.(\d+)$", name) else 0), name),
        )
        for member_name in members:
            for raw_b in zf.read(member_name).splitlines():
                if not raw_b.strip():
                    continue
                line_str = raw_b.decode("latin1").rstrip("\r\n")
                parts = line_str.split("\t")
                ts = datetime.strptime(parts[0].strip(), "%d/%m/%Y %H:%M:%S")
                msg_clean = sanitize_message_text(parts[4])
                raw_lines_sanitized.append(
                    f"{parts[0].strip()}\t{parts[1].strip()}\t{parts[2].strip()}\t{parts[3].strip()}\t{msg_clean}"
                )
                rows.append({
                    "election_year": election_year,
                    "turn": turn,
                    "uf": uf,
                    "cd_pleito": cd_pleito,
                    "cd_municipio": cd_municipio,
                    "nr_zona": nr_zona,
                    "nr_secao": nr_secao,
                    "source_jez_file": Path(jez_filename).name,
                    "source_log_member": member_name,
                    "file_seq": seq,
                    "raw_timestamp": ts,
                    "log_level": parts[1].strip(),
                    "urna_serial": parts[2].strip(),
                    "application": parts[3].strip(),
                    "message_text": msg_clean,
                })
                seq += 1

    return raw_lines_sanitized, pl.DataFrame(rows, schema=FACT_URNA_LOGS_SCHEMA)

In [3]:
# 2. Extração do .jez real de 2026 e visualização de HEAD e TAIL do arquivo original (logd.dat)
jez_filename, jez_bytes = fetch_single_jez_from_remote_zip(TSE_2026_AC_1T_URL, jez_index=0)
raw_lines, df_urna_logs = parse_jez_to_fact_urna_logs(jez_filename, jez_bytes, election_year=2026, turn=1)

print(f"Arquivo .jez extraído: {jez_filename} ({len(jez_bytes):,} bytes comprimidos)")
print(f"Total de linhas no logd.dat: {len(raw_lines):,} | Linhas em fact_urna_logs: {df_urna_logs.height:,}\n")

print("=== HEAD (15 primeiras linhas de logd.dat) ===")
for i, ln in enumerate(raw_lines[:15], start=1):
    print(f"{i:4d} | {ln}")

print("\n=== TAIL (15 últimas linhas de logd.dat) ===")
for i, ln in enumerate(raw_lines[-15:], start=len(raw_lines) - 14):
    print(f"{i:4d} | {ln}")

Arquivo .jez extraído: o03220ac0101500020088-log.jez (68,091 bytes comprimidos)
Total de linhas no logd.dat: 3,714 | Linhas em fact_urna_logs: 3,714

=== HEAD (15 primeiras linhas de logd.dat) ===
   1 | 23/09/2026 10:39:02	INFO	02263093	LOGD	Início das operações do logd
   2 | 23/09/2026 10:39:02	INFO	02263093	LOGD	Urna ligada em 23/09/2026 às 10:38:08
   3 | 23/09/2026 10:39:02	INFO	02263093	SCUE	Iniciando aplicação - 1º turno
   4 | 23/09/2026 10:39:02	INFO	02263093	SCUE	Versão da aplicação: 10.23.0.0 - Praia da Barra do Cahy
   5 | 23/09/2026 10:39:04	INFO	02263093	SCUE	Urna operando com rede elétrica
   6 | 23/09/2026 10:39:04	INFO	02263093	SCUE	Bateria interna com carga plena
   7 | 23/09/2026 10:39:10	INFO	02263093	SCUE	Tamanho da mídia interna: 3.7 GB
   8 | 23/09/2026 10:39:13	INFO	02263093	SCUE	Tamanho da memória: 3.7 GB
   9 | 23/09/2026 10:39:14	INFO	02263093	SCUE	Verificação de assinatura de aplicação por etapa [1] - [Pacote: /bin/avbin.vst] - [SUCESSO]
  10 | 23/09/2026 1

In [4]:
# 3. Inspeção de HEAD (15) e TAIL (15) da tabela fact_urna_logs
print("=== HEAD (15) de fact_urna_logs ===")
print(df_urna_logs.head(15))

print("\n=== TAIL (15) de fact_urna_logs ===")
print(df_urna_logs.tail(15))

print("\n=== Amostra de eventos do aplicativo VOTA durante a votação (file_seq 706..722) ===")
print(
    df_urna_logs.filter(pl.col("file_seq").is_between(706, 722)).select(
        ["file_seq", "raw_timestamp", "log_level", "urna_serial", "application", "message_text"]
    )
)

=== HEAD (15) de fact_urna_logs ===
shape: (15, 15)
┌─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬────────┬───────┬───────┬───────┐
│ ele ┆ tur ┆ uf  ┆ cd_ ┆ cd_ ┆ nr_ ┆ nr_ ┆ sou ┆ sou ┆ fil ┆ raw ┆ log_le ┆ urna_ ┆ appli ┆ messa │
│ cti ┆ n   ┆ --- ┆ ple ┆ mun ┆ zon ┆ sec ┆ rce ┆ rce ┆ e_s ┆ _ti ┆ vel    ┆ seria ┆ catio ┆ ge_te │
│ on_ ┆ --- ┆ str ┆ ito ┆ ici ┆ a   ┆ ao  ┆ _je ┆ _lo ┆ eq  ┆ mes ┆ ---    ┆ l     ┆ n     ┆ xt    │
│ yea ┆ i8  ┆     ┆ --- ┆ pio ┆ --- ┆ --- ┆ z_f ┆ g_m ┆ --- ┆ tam ┆ str    ┆ ---   ┆ ---   ┆ ---   │
│ r   ┆     ┆     ┆ i32 ┆ --- ┆ i16 ┆ i32 ┆ ile ┆ emb ┆ i64 ┆ p   ┆        ┆ str   ┆ str   ┆ str   │
│ --- ┆     ┆     ┆     ┆ i32 ┆     ┆     ┆ --- ┆ er  ┆     ┆ --- ┆        ┆       ┆       ┆       │
│ i16 ┆     ┆     ┆     ┆     ┆     ┆     ┆ str ┆ --- ┆     ┆ dat ┆        ┆       ┆       ┆       │
│     ┆     ┆     ┆     ┆     ┆     ┆     ┆     ┆ str ┆     ┆ eti ┆        ┆       ┆       ┆       │
│     ┆     ┆     ┆     ┆     ┆     ┆  

In [5]:
# 4. Persistência em Parquet ZSTD e verificação de integridade física via DuckDB
out_dir = Path(STAGING_PATH) / "poc_fact_urna_logs"
parquet_path = out_dir / "election_year=2026" / "turn=1" / "uf=AC" / "part-00000.parquet"
parquet_path.parent.mkdir(parents=True, exist_ok=True)
df_urna_logs.write_parquet(parquet_path, compression="zstd", compression_level=5)

pq_meta = pq.read_metadata(parquet_path)
parquet_size = parquet_path.stat().st_size
reduction_pct = (1.0 - parquet_size / len(jez_bytes)) * 100.0
print(
    f"Parquet ZSTD gravado: {parquet_path}\n"
    f"Tamanho Parquet ZSTD: {parquet_size:,} bytes vs {len(jez_bytes):,} bytes do .jez original "
    f"(redução de {reduction_pct:.1f}% sobre o arquivo já comprimido!)"
)

con = duckdb.connect()
cursor = con.execute(
    f"""
    SELECT
        application,
        log_level,
        COUNT(*) AS line_count,
        MIN(file_seq) AS min_seq,
        MAX(file_seq) AS max_seq,
        MIN(raw_timestamp) AS first_ts,
        MAX(raw_timestamp) AS last_ts
    FROM read_parquet('{parquet_path.as_posix()}')
    GROUP BY application, log_level
    ORDER BY line_count DESC
    """
)
cols = [d[0] for d in cursor.description]
summary_df = pl.DataFrame(cursor.fetchall(), schema=cols, orient="row")
print("\n=== Resumo de Integridade por Aplicativo e Nível de Log (DuckDB) ===")
print(summary_df)

# Validações de integridade da ingestão
assert df_urna_logs.height == len(raw_lines) == pq_meta.num_rows
assert df_urna_logs["file_seq"].to_list() == list(range(1, len(raw_lines) + 1))
assert df_urna_logs["raw_timestamp"].null_count() == 0

validation_summary = {
    "status": "SUCCESS",
    "table": "fact_urna_logs",
    "source_jez_file": jez_filename,
    "jez_compressed_bytes": len(jez_bytes),
    "parquet_zstd_bytes": parquet_size,
    "compression_reduction_pct": round(reduction_pct, 2),
    "total_rows": int(df_urna_logs.height),
    "null_timestamps": int(df_urna_logs["raw_timestamp"].null_count()),
}
(Path(STAGING_PATH) / "poc_validation_summary.json").write_text(json.dumps(validation_summary, indent=2), encoding="utf-8")
print("\nResumo de validação salvo em poc_validation_summary.json:")
print(json.dumps(validation_summary, indent=2))

# Validação de higiene pré-upload e publicação condicional via KaggleHub SDK
PHASE_ID = "01_urna_logs_poc"
PHASE_SUMMARY = "PoC de leitura e parsing 1:1 de logs da urna (fact_urna_logs) em Parquet ZSTD"

forbidden_extensions = {".zip", ".jez", ".tmp", ".part", ".log"}
offending_files = [
    str(p.relative_to(DATASET_PATH))
    for p in Path(DATASET_PATH).rglob("*")
    if p.is_file() and any(sfx.lower() in forbidden_extensions for sfx in p.suffixes)
]
if offending_files:
    raise ValueError(
        f"Upload abortado! Arquivos brutos/temporários detectados em DATASET_PATH: {offending_files[:10]}"
    )

if (
    globals().get("__name__") == "__main__"
    and not os.getenv("PYTEST_CURRENT_TEST")
    and os.getenv("TSE_PUBLISH_POC") == "1"
):
    import kagglehub

    if is_directory_empty(Path(DATASET_PATH)):
        raise RuntimeError(
            f"Upload abortado! DATASET_PATH ({DATASET_PATH}) está vazio após o processamento."
        )
    print(f"Publicando nova versão do dataset '{DATASET_HANDLE}' a partir de {DATASET_PATH}...")
    kagglehub.dataset_upload(
        handle=DATASET_HANDLE,
        local_dataset_dir=str(DATASET_PATH),
        version_notes=f"[{PHASE_ID}] {PHASE_SUMMARY}",
        ignore_patterns=[
            "*.tmp",
            "*.part",
            "*.zip",
            "*.jez",
            "*.log",
            "__pycache__/",
            ".ipynb_checkpoints/",
            ".DS_Store",
        ],
    )
    print(f"[SUCESSO] Nova versão de '{DATASET_HANDLE}' publicada no Kaggle!")
else:
    print("[POC] Artefatos de PoC validados em STAGING_PATH (publicação em DATASET_PATH desabilitada por padrão; defina TSE_PUBLISH_POC=1 para publicar).")

Parquet ZSTD gravado: /tmp/fact_urna_logs_luzdslml/election_year=2026/turn=1/uf=AC/part-00000.parquet
Tamanho Parquet ZSTD: 33,768 bytes vs 68,091 bytes do .jez original (redução de 50.4% sobre o arquivo já comprimido!)

=== Resumo de Integridade por Aplicativo e Nível de Log (DuckDB) ===
shape: (8, 7)
┌─────────────┬───────────┬────────────┬─────────┬─────────┬───────────────┬───────────────────────┐
│ application ┆ log_level ┆ line_count ┆ min_seq ┆ max_seq ┆ first_ts      ┆ last_ts               │
│ ---         ┆ ---       ┆ ---        ┆ ---     ┆ ---     ┆ ---           ┆ ---                   │
│ str         ┆ str       ┆ i64        ┆ i64     ┆ i64     ┆ datetime[μs]  ┆ datetime[μs]          │
╞═════════════╪═══════════╪════════════╪═════════╪═════════╪═══════════════╪═══════════════════════╡
│ VOTA        ┆ INFO      ┆ 3161       ┆ 334     ┆ 3714    ┆ 2026-09-23    ┆ 2026-10-04 15:05:11   │
│             ┆           ┆            ┆         ┆         ┆ 10:58:10      ┆              